# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides an interactive guide for loading and exploring the [A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is loaded from a Croissant schema available by URL.

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR² Kilifi Mental Health Survey dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Get and pretty print dataset metadata
metadata = dataset.metadata
print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)
print("Version: ", metadata.version)
print("License: ", metadata.license)


## 2. Data Overview
Review available record sets, fields, and their `@id`s (unique entity identifiers).

*Tip*: In Croissant/`mlcroissant`, record sets, fields, and columns are referenced by their `@id` attributes.

List available record sets in the dataset, enumerate their fields, and examine their IDs for use in data extraction.


In [ ]:
# List all available record sets, their fields, and example fields IDs

print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print("- Record Set @id:", rs.id)
    print("  Name:", rs.name)
    print("  Description:", rs.description)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}  (name: {field.name}, type: {field.data_type})")
    print()


## 3. Data Extraction
Load tabular survey data from a specific record set into a DataFrame for analysis.

_**Note:** Use the record set and field `@id`s identified above. Typically, survey tabular data will be in a record set such as `cr:RecordSet_1` (actual `@id`s may differ—see above cell's output for precise @ids)._

In [ ]:
# Choose the primary record set for the survey data (replace with your dataset's actual @id)
main_record_set_id = None
for rs in dataset.record_sets:
    # Typically, the main survey data is first record set
    if rs.fields:
        main_record_set_id = rs.id
        break

if main_record_set_id is None:
    raise RuntimeError("Couldn't identify a main record set. Please check available record sets.")
print(f"Main record set id: {main_record_set_id}\n")

# Load records from the main record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print("Columns detected in DataFrame (field @id):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter, normalize, and group data using `@id`-based column names.

*We'll select numeric fields (e.g., GAD-7, PHQ-9 scores) by their field `@id`, filter out low values, normalize, and group by a demographic field's `@id` (e.g., `level_of_education` if present).*


In [ ]:
# Identify common numeric fields and groupable demographic fields by their @id

# Example field IDs (check above for actual dataset IDs!):
# e.g. 'GAD7_score_id' = '@id:field:cr:GAD7_score', 'level_of_education_id' = '@id:field:cr:level_of_education'
# Find them in df.columns by keywords:
numeric_field_id = None
group_field_id = None

for c in df.columns:
    # Often, mental health score fields will contain 'GAD' or 'PHQ' or 'score' in their id/name
    if 'gad' in c.lower() and ('score' in c.lower() or 'total' in c.lower()):
        numeric_field_id = c
    if 'education' in c.lower():
        group_field_id = c
if numeric_field_id is None:
    # Fallback: Pick the first numeric column
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
if group_field_id is None:
    # Fallback: pick age or gender if not found
    for c in df.columns:
        if 'age' in c.lower():
            group_field_id = c
            break

print(f"Numeric field ID used: {numeric_field_id}")
print(f"Group field ID used: {group_field_id}\n")

# Drop missing data for this column
df = df.dropna(subset=[numeric_field_id])

# Basic EDA
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Records with {numeric_field_id} above the mean ({threshold:.2f}): {len(filtered_df)} records\n")

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print("Sample with normalized field:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group analysis (if group_field_id is present)
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped)
else:
    print(f"Group field {group_field_id} not found in columns.")

## 5. Visualization
Visualize the distribution of a numeric survey score field (e.g., GAD-7) and compare across groups if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- Successfully loaded and explored a FAIR² Croissant dataset using Python and `mlcroissant`.
- Explored the dataset's available record sets and fields using their machine-actionable `@id`.
- Loaded main survey records into a Pandas DataFrame for analysis.
- Performed basic filtering, normalization, aggregation, and visualization using field `@id` for safe referencing.

**Next steps:** This dataset can now be used for richer health or demographic analyses, predictive modeling, and integration with other Croissant-compatible datasets!